KUL H02A5a Computer Vision: Group Assignment 2
---------------------------------------------------------------

**Group 6**

Student names: Kaixi Yao, Enmin Lin, Taicheng Liu, Zhenyang Li.

In this group assignment your team will delve into some deep learning applications for computer vision. The assignment will be delivered in the same groups from *Group assignment 1* and you start from this template notebook. You can make use of the *Group assignment 2* forum/discussion board on Toledo if you have any questions.

The notebook you submit for grading is the last notebook pinned as default and submitted to the competition prior to the deadline.

Good luck and have fun!

---------------------------------------------------------------
NOTES:
* This notebook is just a template. Please keep the five main sections, but feel free to adjust further in any way you please!
* Clearly indicate the improvements that you make! You can for instance use subsections like: *3.1. Improvement: applying loss function f instead of g*.


# Overview
This assignment consists of *three main parts* for which we expect you to provide code and extensive documentation in the notebook, with a final discussion:
* Image classification (Sect. 1)
* Semantic segmentation (Sect. 2)
* Adversarial attacks (Sect. 3)
* Discussion (Sect. 4)

## Deep learning resources
If you did not yet explore this in *Group assignment 1 (Sect. 2)*, we recommend using the Pytorch or TensorFlow (and/or Keras) library for building deep learning models.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
import numpy as np
import pandas as pd

# Uncomment the one you'll use
#import tensorflow as tf
#import torch

from matplotlib import pyplot as plt

## PASCAL VOC 2009
For this project you will be using the [PASCAL VOC 2009](http://host.robots.ox.ac.uk/pascal/VOC/voc2009/index.html) dataset. This dataset consists of colour images of various scenes with different object classes (e.g. animal: *bird, cat, ...*; vehicle: *aeroplane, bicycle, ...*), totalling 20 classes.

In [ ]:
# Loading the training data
train_df = pd.read_csv('/kaggle/input/kul-computer-vision-ga-2-2025/train/train_set.csv', index_col="Id")
labels = train_df.columns
train_df["img"] = [np.load('/kaggle/input/kul-computer-vision-ga-2-2025/train/img/train_{}.npy'.format(idx)) for idx, _ in train_df.iterrows()]
train_df["seg"] = [np.load('/kaggle/input/kul-computer-vision-ga-2-2025/train/seg/train_{}.npy'.format(idx)) for idx, _ in train_df.iterrows()]
print("The training set contains {} examples.".format(len(train_df)))

# Show some examples
fig, axs = plt.subplots(2, 20, figsize=(10 * 20, 10 * 2))
for i, label in enumerate(labels):
    df = train_df.loc[train_df[label] == 1]
    axs[0, i].imshow(df.iloc[0]["img"], vmin=0, vmax=255)
    axs[0, i].set_title("\n".join(label for label in labels if df.iloc[0][label] == 1), fontsize=40)
    axs[0, i].axis("off")
    axs[1, i].imshow(df.iloc[0]["seg"], vmin=0, vmax=20)  # with the absolute color scale it will be clear that the arrays in the "seg" column are label maps (labels in [0, 20])
    axs[1, i].axis("off")
    
plt.show()

# The training dataframe contains for each image 20 columns with the ground truth classification labels and 20 column with the ground truth segmentation maps for each class
train_df.head(1)

In [ ]:
# Loading the test data
test_df = pd.read_csv('/kaggle/input/kul-computer-vision-ga-2-2025/test/test_set.csv', index_col="Id")
test_df["img"] = [np.load('/kaggle/input/kul-computer-vision-ga-2-2025/test/img/test_{}.npy'.format(idx)) for idx, _ in test_df.iterrows()]
test_df["seg"] = [-1 * np.ones(img.shape[:2], dtype=np.int8) for img in test_df["img"]]
print("The test set contains {} examples.".format(len(test_df)))

# The test dataframe is similar to the training dataframe, but here the values are -1 --> your task is to fill in these as good as possible in Sect. 2 and Sect. 3; in Sect. 6 this dataframe is automatically transformed in the submission CSV!
test_df.head(1)

## Your Kaggle submission
Your filled test dataframe (during Sect. 2 and Sect. 3) must be converted to a submission.csv with two rows per example (one for classification and one for segmentation) and with only a single prediction column (the multi-class/label predictions running length encoded). You don't need to edit this section. Just make sure to call this function at the right position in this notebook.

In [ ]:
def _rle_encode(img):
    """
    Kaggle requires RLE encoded predictions for computation of the Dice score (https://www.kaggle.com/lifa08/run-length-encode-and-decode)

    Parameters
    ----------
    img: np.ndarray - binary img array
    
    Returns
    -------
    rle: String - running length encoded version of img
    """
    pixels = img.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    rle = ' '.join(str(x) for x in runs)
    return rle

def generate_submission(df):
    """
    Make sure to call this function once after you completed Sect. 2 and Sect. 3! It transforms and writes your test dataframe into a submission.csv file.
    
    Parameters
    ----------
    df: pd.DataFrame - filled dataframe that needs to be converted
    
    Returns
    -------
    submission_df: pd.DataFrame - df in submission format.
    """
    df_dict = {"Id": [], "Predicted": []}
    for idx, _ in df.iterrows():
        df_dict["Id"].append(f"{idx}_classification")
        df_dict["Predicted"].append(_rle_encode(np.array(df.loc[idx, labels])))
        df_dict["Id"].append(f"{idx}_segmentation")
        df_dict["Predicted"].append(_rle_encode(np.array([df.loc[idx, "seg"] == j + 1 for j in range(len(labels))])))
    
    submission_df = pd.DataFrame(data=df_dict, dtype=str).set_index("Id")
    submission_df.to_csv("submission.csv")
    return submission_df

# 1. Image classification
The goal here is simple: implement a classification model and train it to recognise all 20 classes (and/or background) using the training set and compete on the test set (by filling in the classification columns in the test dataframe).

## Final classification model and results

For the final classification pipeline, we did not use the random baseline included in the template notebook. Instead, the classification predictions are produced by the code in `Image classification/`, with one isolated output directory per experiment under `output/image_classification/<experiment>/`.

The best submitted classifier is `convnext_small_320`, a ConvNeXt-Small backbone initialized with ImageNet-1K weights and fine-tuned for 20-label PASCAL VOC classification. The model uses 320 x 320 input resolution, AsymmetricLoss, per-class threshold search, a three-stage training schedule, and horizontal-flip test-time augmentation.

Summary of the main classification experiments:

| Version | Experiment | Backbone | Input | val mAP | Kaggle display | Adjusted classification Dice |
|---|---|---|---|---:|---:|---:|
| v1.1 | `resnet50_224` | ResNet-50 | 224 + TTA | 0.8175 | 0.39165 | 0.78330 |
| v2 | `efficientnet_b3_320` | EfficientNet-B3 | 320 + TTA | 0.8599 | 0.42813 | 0.85626 |
| v3 | `convnext_tiny_320` | ConvNeXt-Tiny | 320 + TTA | 0.8933 | 0.43673 | 0.87346 |
| v4 | `convnext_small_320` | ConvNeXt-Small | 320 + TTA | 0.8995 | 0.44905 | 0.89810 |

The latest complete submission combines `convnext_small_320` classification with the v10 segmentation output (`submission_exp_v10_segman_b_iter25000.csv`) and obtains an overall Kaggle score of **0.87588**. The classification-only Kaggle display score for `convnext_small_320` is **0.44905**; under our classification-only comparison convention this corresponds to an adjusted classification Dice of **0.89810** because half of the rows in a classification-only submission are empty segmentation placeholders.

Important generated files:

- `output/image_classification/convnext_small_320/metrics/evaluation_summary.csv`
- `output/image_classification/convnext_small_320/metrics/ap_per_class.csv`
- `output/image_classification/convnext_small_320/predictions/test_probabilities_convnext_small_320.csv`
- `output/image_classification/convnext_small_320/predictions/test_binary_predictions_convnext_small_320.csv`
- `output/image_classification/convnext_small_320/submissions/submission_classification_convnext_small_320.csv`
- `output/image_classification/convnext_small_320/submissions/submission_final_convnext_small_320__seg_submission_exp_v10_segman_b_iter25000.csv`


In [ ]:
class RandomClassificationModel:
    """
    Random classification model: 
        - generates random labels for the inputs based on the class distribution observed during training
        - assumes an input can have multiple labels
    """
    def fit(self, X, y):
        """
        Adjusts the class ratio variable to the one observed in y. 

        Parameters
        ----------
        X: list of arrays - n x (height x width x 3)
        y: list of arrays - n x (nb_classes)

        Returns
        -------
        self
        """
        self.distribution = np.mean(y, axis=0)
        print("Setting class distribution to:\n{}".format("\n".join(f"{label}: {p}" for label, p in zip(labels, self.distribution))))
        return self
        
    def predict(self, X):
        """
        Predicts for each input a label.
        
        Parameters
        ----------
        X: list of arrays - n x (height x width x 3)
            
        Returns
        -------
        y_pred: list of arrays - n x (nb_classes)
        """
        np.random.seed(0)
        return [np.array([int(np.random.rand() < p) for p in self.distribution]) for _ in X]
    
    def __call__(self, X):
        return self.predict(X)
    
model = RandomClassificationModel()
model.fit(train_df["img"], train_df[labels])
test_df.loc[:, labels] = model.predict(test_df["img"])
test_df.head(1)

# 2. Semantic segmentation
The goal here is to implement a segmentation model that labels every pixel in the image as belonging to one of the 20 classes (and/or background). Use the training set to train your model and compete on the test set (by filling in the segmentation column in the test dataframe).

In [ ]:
class RandomSegmentationModel:
    """
    Random segmentation model: 
        - generates random label maps for the inputs based on the class distributions observed during training
        - every pixel in an input can only have one label
    """
    def fit(self, X, Y):
        """
        Adjusts the class ratio variable to the one observed in Y. 

        Parameters
        ----------
        X: list of arrays - n x (height x width x 3)
        Y: list of arrays - n x (height x width)

        Returns
        -------
        self
        """
        self.distribution = np.mean([[np.sum(Y_ == i) / Y_.size for i in range(len(labels) + 1)] for Y_ in Y], axis=0)
        print("Setting class distribution to:\nbackground: {}\n{}".format(self.distribution[0], "\n".join(f"{label}: {p}" for label, p in zip(labels, self.distribution[1:]))))
        return self
        
    def predict(self, X):
        """
        Predicts for each input a label map.
        
        Parameters
        ----------
        X: list of arrays - n x (height x width x 3)
            
        Returns
        -------
        Y_pred: list of arrays - n x (height x width)
        """
        np.random.seed(0)
        return [np.random.choice(np.arange(len(labels) + 1), size=X_.shape[:2], p=self.distribution) for X_ in X]
    
    def __call__(self, X):
        return self.predict(X)
    
model = RandomSegmentationModel()
model.fit(train_df["img"], train_df["seg"])
test_df.loc[:, "seg"] = model.predict(test_df["img"])
test_df.head(1)

## Submit to competition
You don't need to edit this section. Just use it at the right position in the notebook. See the definition of this function in Sect. 1.3 for more details.

In [ ]:
generate_submission(test_df)

# 3. Adversarial attack

For the adversarial part, we attack the final image-classification model rather than the segmentation model. We use the best single classification model, `convnext_small_320`, and keep all classifier weights frozen. The task is a targeted white-box attack: for validation images whose ground-truth label does **not** contain `aeroplane`, we add a small perturbation so that the fixed classifier becomes confident that `aeroplane` is present.

This is a white-box setting because the adversary has access to the classifier and its gradients. We use the validation split instead of the Kaggle test set because the test labels are unavailable; filtering to target-negative validation images gives a clear success criterion.

The implementation is provided in `Image classification/07_adversarial_attack.py`. It loads `output/image_classification/convnext_small_320/checkpoints/final_model.pth`, applies the same validation preprocessing as the classifier, and runs two gradient-based attacks:

- **FGSM**: one gradient-sign step that minimizes the binary cross-entropy loss for the target class `aeroplane`.
- **PGD**: repeated projected gradient-sign steps under the same L-infinity perturbation bound.

The classifier parameters stay frozen. Only the input image is differentiated. The regularization is an explicit perturbation constraint: `epsilon=0.03` in normalized image space, followed by clamping to valid image bounds. In pixel space this corresponds to a mean L-infinity change of about `0.00687` on the `[0, 1]` image scale, which is small enough that the clean and adversarial images remain visually very similar.

Final command:

```bash
python "Image classification/07_adversarial_attack.py" --experiment convnext_small_320 --target-class aeroplane --max-samples 64 --epsilon 0.03 --alpha 0.01 --pgd-steps 10 --ckpt final
```

Final result on 64 target-negative validation images, using the learned `aeroplane` decision threshold of `0.60`:

| Attack | Target | epsilon | alpha | steps | Samples | Clean target prob | Adversarial target prob | Clean activation | Adversarial activation | Pixel L-inf mean |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| FGSM | aeroplane | 0.03 | - | 1 | 64 | 0.074 | 0.438 | 0.000 | 0.266 | 0.00687 |
| PGD | aeroplane | 0.03 | 0.01 | 10 | 64 | 0.074 | 0.983 | 0.000 | 1.000 | 0.00687 |

PGD is substantially stronger than FGSM: it pushed every attacked target-negative image above the aeroplane decision threshold, while FGSM succeeded on 26.6% of the images. The clean images had 0% aeroplane activation, so the effect is caused by the perturbation rather than by already-positive target labels.

![Adversarial aeroplane examples](output/image_classification/convnext_small_320/figures/adversarial_aeroplane_examples.png)

The middle column shows the perturbation amplified by 10x; the actual adversarial image on the right remains recognizable to a human observer and looks nearly identical to the clean image. This demonstrates a real local weakness of the classifier: with gradient access, a very small targeted perturbation can strongly change the model's multi-label prediction.

This is a realistic diagnostic for white-box robustness, but it is not yet a realistic remote black-box attack because it assumes access to model weights and gradients. It also does not mean the classifier is useless: the original model still performs well on normal validation and Kaggle images. The result means that the model is not robust to worst-case local perturbations. In general, robustness could be improved with adversarial training, stronger augmentation, input preprocessing or detection, model ensembles, and evaluation under both white-box and black-box attacks.


In [ ]:
# Optional: reproduce the final adversarial-attack experiment locally.
# This requires the trained convnext_small_320 checkpoint.
!python "Image classification/07_adversarial_attack.py" --experiment convnext_small_320 --target-class aeroplane --max-samples 64 --epsilon 0.03 --alpha 0.01 --pgd-steps 10 --ckpt final


# 4. Discussion

## Backbone and Transfer Learning

The main constraint in the image classification task is the small scale of the
training set. The available training split contains 749 labelled images, and
each image has on average only 1.43 positive class labels. Training a deep
convolutional network from scratch would therefore be severely underdetermined:
the model would need to learn both low-level visual filters and high-level class
semantics from only a few hundred examples. For this reason, we relied on
ImageNet-pretrained backbones and fine-tuned them for the PASCAL VOC multi-label
setting.

This choice follows the transfer learning principles discussed in the course.
Early convolutional layers learn generic edge, colour, texture and shape
features, while later layers become increasingly task-specific. Since ImageNet
contains many object categories that overlap conceptually with PASCAL VOC, such
as person, cat, dog, bicycle, car and aeroplane, the pretrained representations
provide a useful starting point. We then replaced the original classifier with a
20-dimensional multi-label head and fine-tuned the network using sigmoid outputs
instead of a softmax, because multiple objects can be present in the same image.

Our experiments show a clear benefit from stronger pretrained backbones and
higher input resolution. ResNet-50 at 224 x 224 reached a validation mAP of
approximately 0.818 after the loss and training improvements. EfficientNet-B3 at
320 x 320 improved this to about 0.860. ConvNeXt-Tiny at 320 x 320 further
improved the validation mAP to 0.893 and obtained a Kaggle displayed score of
0.43673, corresponding to an adjusted classification Dice of 0.87346 under our
classification-only comparison convention. A later ConvNeXt-Small experiment
improved both the local validation mAP and the Kaggle score: it reached
validation mAP 0.8995 and a Kaggle displayed classification score of 0.44905,
corresponding to an adjusted classification Dice of 0.89810. The complete
submission that combined ConvNeXt-Small classification with the v10
segmentation output obtained an overall Kaggle score of 0.87588.

## Augmentation and Training Strategy

The dataset is both small and imbalanced. The most common class, person, appears
207 times, while rare classes such as sheep and cow appear only 27 and 30 times.
Several other classes, including bus, bicycle and train, have around 40 positive
examples. This makes the classifier vulnerable to overfitting and to learning
class-specific shortcuts from the small training set.

To reduce this risk, we used a moderate augmentation pipeline consisting of
random horizontal flips, random rotations, colour jitter and random erasing.
These transformations preserve the semantic labels while changing pose, colour,
illumination and local visibility. They are especially relevant for PASCAL VOC,
where objects can appear at different scales, positions and backgrounds. We did
not use MixUp in the final pipeline, because the multi-label setting and noisy
labels already make the interpretation of soft targets less direct.

The final training procedure used three stages. First, we froze the backbone and
trained only the classification head. Second, we unfroze the full network and
fine-tuned it with a smaller learning rate. Third, we reloaded the best
validation checkpoint and trained on all available labelled images. This last
stage deliberately sacrifices the validation split in exchange for using all
training examples before test prediction. It is useful for the final Kaggle
submission, but it also means that the final checkpoint no longer has an
independent validation estimate. For analysis, we therefore report validation
mAP from the best Stage 2 checkpoint.

## Loss Function and Noisy Labels

A key difficulty in this assignment is that absence from the annotation table
does not always mean visual absence from the image. For example, people,
dining tables, bottles or plants may appear in the background without being
annotated as positive labels. Standard binary cross-entropy treats every
unlabelled class as a true negative, so it can penalise the model for detecting
objects that are visually present but missing from the labels.

To address this, we used AsymmetricLoss instead of standard BCE. Its negative
focal term strongly down-weights easy negative examples, while the clipping term
reduces the contribution of likely false negatives. This matches the structure
of the VOC task: positive labels should remain informative, but some negative
labels are uncertain. The improvement is supported by the experiments, although
not as a perfectly isolated ablation. The original ResNet-50 pipeline with
NegativeSmoothBCE obtained a Kaggle display score of 0.38084. After switching to
AsymmetricLoss and improving the training/prediction pipeline, the ResNet-50
experiment reached 0.39165. Stronger backbones with the same ASL-based training
strategy then improved further: EfficientNet-B3 reached 0.42813 and
ConvNeXt-Tiny reached 0.43673 on the Kaggle display score.

## mAP, Dice and Thresholds

We used validation mAP to compare model ranking quality, but Kaggle evaluates a
Dice score on binarised predictions. These metrics answer different questions.
mAP measures whether positives are ranked above negatives over all possible
thresholds, while Dice/F1 depends on one selected threshold per class. A model
can have high mAP and still perform poorly on Kaggle if the thresholds are badly
calibrated.

For this reason, we performed per-class threshold search on the validation set.
This is closer to the Kaggle objective, because the classification output is a
binary vector that is run-length encoded. The threshold search is especially
important for imbalanced classes: rare classes often need different decision
thresholds from frequent ones. In real applications this distinction also
matters. A retrieval system might care more about ranking quality, while an
automatic tagging system or a safety-critical detector needs calibrated binary
decisions and must explicitly manage false positives and false negatives.

## Per-Class Behaviour and Failure Cases

The per-class results show that the model performs best on visually distinctive
object classes. For ConvNeXt-Small, classes such as bicycle, bus, cow, bird and
train reached AP values close to 1.0 on the validation split. These classes tend
to have distinctive global shapes and backgrounds, which makes them easier for a
classification backbone to identify.

The weakest classes were diningtable, pottedplant, sofa, bottle and sheep.
Diningtable remained the hardest class, with AP around 0.557 for
ConvNeXt-Small. This is likely caused by a combination of label noise and visual
ambiguity: tables often appear as partially visible background objects, and
their appearance changes heavily depending on viewpoint and occlusion. Bottles
and potted plants are small objects, so resizing the full image to 320 x 320 can
still remove important details. Sofas and chairs can be confused with each
other or with other indoor furniture. Sheep is rare in the training data, with
only 27 positive examples, so the classifier has fewer opportunities to learn
robust variations.

Possible improvements would therefore not only involve larger backbones. For
small objects, higher input resolution, object crops or detection-style
pretraining could help. For noisy classes such as diningtable, manual inspection
or semi-supervised relabelling could reduce false-negative supervision. For rare
classes, class-balanced sampling or targeted augmentation may help, although
they would need to be validated carefully to avoid overfitting.

## Real-World Relevance and Limitations

The transfer learning approach is practical and effective, but it also has
limitations. ImageNet pretraining transfers useful visual features, yet the
model remains tied to the distribution of the training set. PASCAL VOC images
are natural photographs with a limited set of 20 object classes. A classifier
trained on this data may fail under domain shift, for example on medical images,
surveillance footage, low-light scenes, unusual camera angles or objects from
non-Western environments. This is a dataset bias issue rather than only a model
capacity issue.

The model is also not suitable for safety-critical deployment without further
calibration and testing. A high Kaggle score does not guarantee reliable
behaviour on rare cases. Missing a small bottle, plant or person can be
acceptable in a benchmark, but not in applications such as robotics, driving or
industrial inspection. In those settings, the cost of different mistakes must be
defined explicitly, and the threshold should be chosen based on the application
rather than only validation F1.

Finally, increasing model size has diminishing returns on this dataset.
ConvNeXt-Small improved over ConvNeXt-Tiny, but it also overfit quickly: its
best validation loss occurred early in Stage 2, while later epochs continued to
reduce training loss but worsened validation loss. ConvNeXt-Base and
ConvNeXt-Large are technically feasible on stronger GPUs, but the small dataset
means they may improve throughput or representation capacity without improving
generalisation. If more time were available, the most useful next steps would
be to run a carefully controlled threshold comparison, inspect actual
false-positive and false-negative examples visually, and test whether larger
ConvNeXt variants improve Kaggle performance rather than only validation mAP.
